<a href="https://colab.research.google.com/github/svanshika-afk/tinyML-lab-2548544/blob/lab2/2548544_TINYML_LAB2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from google.colab import files

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# ==========================================
# STEP 1: LOAD SENSOR DATA
# ==========================================
csv_files = {'idle': 'idle.csv', 'walking': 'walking.csv', 'situps': 'situps.csv'}

# Check if files exist in current directory, otherwise prompt upload
missing_files = [f for f in csv_files.values() if not os.path.exists(f)]
if missing_files:
    print(f"Please upload the missing CSV files: {missing_files}")
    uploaded = files.upload()

def load_data(file_path):
    # sep=None with python engine handles both comma and tab Phyphox exports
    df = pd.read_csv(file_path, sep=None, engine='python')

    # Locate acceleration columns
    cols = df.columns
    x_col = [c for c in cols if 'x' in c.lower() and 'acc' in c.lower()][0]
    y_col = [c for c in cols if 'y' in c.lower() and 'acc' in c.lower()][0]
    z_col = [c for c in cols if 'z' in c.lower() and 'acc' in c.lower()][0]

    ax = df[x_col].values
    ay = df[y_col].values
    az = df[z_col].values

    # Compute 3D Acceleration Vector Magnitude: a_mag = sqrt(ax^2 + ay^2 + az^2)
    a_mag = np.sqrt(ax**2 + ay**2 + az**2)
    return a_mag

# ==========================================
# STEP 2: WINDOWING & FEATURE EXTRACTION
# ==========================================
WINDOW_SIZE = 100  # 1-second interval at 100 Hz sampling rate
classes = {'idle': 0, 'walking': 1, 'situps': 2}

X_features, y_labels = [], []

for activity, label in classes.items():
    mag_data = load_data(csv_files[activity])
    num_windows = len(mag_data) // WINDOW_SIZE

    for i in range(num_windows):
        window = mag_data[i * WINDOW_SIZE : (i + 1) * WINDOW_SIZE]

        # 1. Mean (μ)
        mean_val = np.mean(window)
        # 2. Standard Deviation (σ)
        std_val = np.std(window)
        # 3. Root Mean Square (RMS)
        rms_val = np.sqrt(np.mean(window**2))

        X_features.append([mean_val, std_val, rms_val])
        y_labels.append(label)

X = np.array(X_features, dtype=np.float32)
y = np.array(y_labels, dtype=np.int32)

print(f"Dataset extracted: {X.shape[0]} windows across 3 classes, 3 features/window.")

# Train/Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize inputs
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ==========================================
# STEP 3: BASELINE MLP MODEL TRAINING
# ==========================================
# Architecture sized to overcome TFLite FlatBuffer metadata overhead
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(3,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n--- Training Baseline Model ---")
history = model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=8,
    validation_data=(X_test, y_test),
    verbose=0
)
test_loss, baseline_keras_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Trained Keras Model Accuracy: {baseline_keras_acc * 100:.2f}%")

# Save Keras Model
model.save("baseline_model.keras")

# ==========================================
# STEP 4: TFLITE QUANTIZATION VARIANTS
# ==========================================
print("\n--- Generating TFLite Quantization Variants ---")

# Representative dataset generator for calibration
def representative_data_gen():
    for i in range(min(100, len(X_train))):
        yield [X_train[i:i+1].astype(np.float32)]

# Variant A: Baseline (Float32)
conv_baseline = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_baseline = conv_baseline.convert()
with open("model_float32.tflite", "wb") as f:
    f.write(tflite_baseline)

# Variant B: Dynamic Range Quantization (Weights int8, Activations float32)
conv_dynamic = tf.lite.TFLiteConverter.from_keras_model(model)
conv_dynamic.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_dynamic = conv_dynamic.convert()
with open("model_dynamic.tflite", "wb") as f:
    f.write(tflite_dynamic)

# Variant C: Float16 Quantization (Weights float16)
conv_fp16 = tf.lite.TFLiteConverter.from_keras_model(model)
conv_fp16.optimizations = [tf.lite.Optimize.DEFAULT]
conv_fp16.target_spec.supported_types = [tf.float16]
tflite_fp16 = conv_fp16.convert()
with open("model_float16.tflite", "wb") as f:
    f.write(tflite_fp16)

# Variant D: Full Integer Quantization (PTQ - Weights int8, Activations int8)
conv_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
conv_int8.optimizations = [tf.lite.Optimize.DEFAULT]
conv_int8.representative_dataset = representative_data_gen
conv_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv_int8.inference_input_type = tf.int8
conv_int8.inference_output_type = tf.int8
tflite_int8 = conv_int8.convert()
with open("model_int8.tflite", "wb") as f:
    f.write(tflite_int8)

# ==========================================
# STEP 5: BENCHMARKING (SIZE, ACCURACY, LATENCY)
# ==========================================
def evaluate_tflite(model_path, X_data, y_data):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    is_int8_io = (input_details['dtype'] == np.int8)

    correct = 0
    latencies = []

    for sample, true_label in zip(X_data, y_data):
        input_tensor = sample.reshape(input_details['shape'])

        # Scale/zero-point quantization for INT8 I/O
        if is_int8_io:
            scale, zero_point = input_details['quantization']
            input_tensor = np.round(input_tensor / scale + zero_point).astype(np.int8)
        else:
            input_tensor = input_tensor.astype(input_details['dtype'])

        interpreter.set_tensor(input_details['index'], input_tensor)

        start_t = time.perf_counter()
        interpreter.invoke()
        end_t = time.perf_counter()

        latencies.append((end_t - start_t) * 1e6)  # microseconds (μs)

        output = interpreter.get_tensor(output_details['index'])
        pred = np.argmax(output)
        if pred == true_label:
            correct += 1

    accuracy = (correct / len(y_data)) * 100.0
    mean_latency_us = np.mean(latencies)
    size_kb = os.path.getsize(model_path) / 1024.0

    return size_kb, accuracy, mean_latency_us

models_to_test = [
    ("Baseline (Unquantized)", "Float32", "model_float32.tflite"),
    ("Dynamic Range", "Weights int8, Act float32", "model_dynamic.tflite"),
    ("Float16 Quantization", "Float16", "model_float16.tflite"),
    ("Full Integer (PTQ)", "Weights int8, Act int8", "model_int8.tflite"),
]

results = []
baseline_size = os.path.getsize("model_float32.tflite") / 1024.0

for scheme, dtype, path in models_to_test:
    size_kb, acc, lat = evaluate_tflite(path, X_test, y_test)
    mem_red = ((baseline_size - size_kb) / baseline_size) * 100.0
    results.append({
        "Quantization Scheme": scheme,
        "Target Data Type": dtype,
        "Model Size (KB)": f"{size_kb:.2f}",
        "Accuracy (%)": f"{acc:.2f}%",
        "Memory Reduction (%)": f"{mem_red:.1f}%",
        "Inference Time (μs)": f"{lat:.1f} μs"
    })

results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("BENCHMARKING RESULTS TABLE")
print("="*80)
print(results_df.to_markdown(index=False))


Dataset extracted: 210 windows across 3 classes, 3 features/window.

--- Training Baseline Model ---
Trained Keras Model Accuracy: 97.62%

--- Generating TFLite Quantization Variants ---
Saved artifact at '/tmp/tmp0jvak5bj'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  140052744751056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140052744752016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140052744751632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140052744752400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140052744752208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140052744752784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140052744750864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140052744753360: TensorSpec(shape=(), dtype=tf.

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



BENCHMARKING RESULTS TABLE
| Quantization Scheme    | Target Data Type          |   Model Size (KB) | Accuracy (%)   | Memory Reduction (%)   | Inference Time (μs)   |
|:-----------------------|:--------------------------|------------------:|:---------------|:-----------------------|:----------------------|
| Baseline (Unquantized) | Float32                   |             45.18 | 97.62%         | 0.0%                   | 41.6 μs               |
| Dynamic Range          | Weights int8, Act float32 |             16.37 | 97.62%         | 63.8%                  | 8.8 μs                |
| Float16 Quantization   | Float16                   |             24.77 | 97.62%         | 45.2%                  | 5.0 μs                |
| Full Integer (PTQ)     | Weights int8, Act int8    |             19.45 | 97.62%         | 56.9%                  | 7.4 μs                |


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
